# 🎙️ OmniVoice Studio — Teste no Google Colab

**Objetivo:** testar o OmniVoice Studio (interface desktop do debpalash) e o OmniVoice puro
para avaliar se vale instalar localmente no Windows.

**GPU:** este notebook requer GPU T4 (gratuita no Colab). Ative em:
`Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4`

---
## ⚠️ Aviso importante
O OmniVoice Studio usa Tauri (desktop app). No Colab, testamos apenas o **OmniVoice puro** via Python
e comparamos com o que o Studio ofereceria. Isso é suficiente para avaliar a qualidade de voz.


## Célula 1 — Verificar GPU disponível

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU não encontrada — ative a GPU nas configurações do Colab!')

import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM total: {vram:.1f} GB')

## Célula 2 — Instalar OmniVoice

In [ ]:
# Instala o OmniVoice (k2-fsa)
!pip install -q omnivoice

# Dependências de áudio
!pip install -q soundfile scipy numpy

# Para reprodução no Colab
!pip install -q IPython

print('✅ Instalação concluída!')

## Célula 3 — Baixar modelo OmniVoice

O modelo (~1.5 GB) é baixado automaticamente na primeira execução.

In [ ]:
import time
from omnivoice import OmniVoice, OmniVoiceGenerationConfig

print('Carregando modelo OmniVoice (primeira vez faz download ~1.5 GB)...')
t0 = time.time()
m = OmniVoice.from_pretrained('k2-fsa/omnivoice')
print(f'✅ Modelo carregado em {time.time()-t0:.1f}s')

## Célula 4 — Teste básico: voz em português

In [ ]:
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

SR = 24000  # sample rate do OmniVoice

texto = 'Olá! Eu sou a Pandora, secretária particular de Hades, senhor do Submundo. Como posso ajudá-lo hoje?'

cfg = OmniVoiceGenerationConfig(num_step=32, guidance_scale=3.0)

print('Gerando áudio...')
t0 = time.time()
wav = m.generate(texto, language='pt', generation_config=cfg)[0]
wav = np.asarray(wav, dtype=np.float32).squeeze()
print(f'✅ Gerado em {time.time()-t0:.1f}s | Duração: {len(wav)/SR:.1f}s')

sf.write('teste_basico.wav', wav, SR)
display(Audio('teste_basico.wav'))

## Célula 5 — Teste com instruct (descrição de voz)

In [ ]:
# Vozes válidas para instruct:
# female, male, child, teenager, young adult, middle-aged, elderly
# low pitch, moderate pitch, high pitch, very high pitch, very low pitch
# whisper
# american accent, british accent, australian accent, indian accent,
# portuguese accent, chinese accent, japanese accent, korean accent, etc.

vozes_teste = [
    ('feminino_moderado', 'female, young adult, moderate pitch'),
    ('masculino_grave',   'male, middle-aged, low pitch'),
    ('sussurro',          'female, whisper'),
    ('crianca',           'child, high pitch'),
    ('idoso_grave',       'male, elderly, very low pitch'),
]

texto_teste = 'Era uma vez, num reino muito distante, uma jovem que descobriu o segredo do universo.'

for nome, instruct in vozes_teste:
    print(f'\n🎙️ [{nome}] instruct: "{instruct}"')
    t0 = time.time()
    wav = m.generate(texto_teste, language='pt', instruct=instruct, generation_config=cfg)[0]
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    fname = f'{nome}.wav'
    sf.write(fname, wav, SR)
    print(f'   Gerado em {time.time()-t0:.1f}s')
    display(Audio(fname))

## Célula 6 — Teste de clonagem de voz

Faça upload de um arquivo WAV de referência (qualquer voz de ~5-30s)
e o OmniVoice tenta clonar o timbre.

In [ ]:
from google.colab import files
import io

print('📁 Faça upload do seu arquivo WAV de referência:')
uploaded = files.upload()

if uploaded:
    fname = list(uploaded.keys())[0]
    print(f'\nArquivo recebido: {fname}')
    
    # Carrega áudio de referência
    ref_audio, ref_sr = sf.read(io.BytesIO(uploaded[fname]))
    if ref_audio.ndim > 1:
        ref_audio = ref_audio.mean(axis=1)  # stereo → mono
    print(f'Referência: {len(ref_audio)/ref_sr:.1f}s @ {ref_sr}Hz')
    
    # Cria o voice clone prompt
    print('Criando embedding de clonagem...')
    prompt = m.create_voice_clone_prompt(
        reference_list=[ref_audio.tolist()],
        sr=ref_sr
    )
    
    # Gera com a voz clonada
    texto_clone = 'Esta é uma demonstração da clonagem de voz. Meu timbre foi copiado a partir do arquivo de referência que você enviou.'
    
    print('Gerando com voz clonada...')
    t0 = time.time()
    wav = m.generate(texto_clone, language='pt', voice_clone_prompt=prompt, generation_config=cfg)[0]
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    print(f'✅ Gerado em {time.time()-t0:.1f}s')
    
    sf.write('clone_resultado.wav', wav, SR)
    display(Audio('clone_resultado.wav'))
else:
    print('Nenhum arquivo enviado. Pule esta célula se não tiver um WAV de referência.')

## Célula 7 — Teste de velocidade: quantos segundos por segundo de áudio?

In [ ]:
textos_benchmark = [
    'Olá.',                                          # frase curtíssima
    'Era uma vez num reino muito distante.',          # frase curta
    'A inteligência artificial está transformando a maneira como interagimos com a tecnologia, abrindo novos horizontes para a criatividade e a produtividade humana.',  # média
]

print('📊 Benchmark de velocidade (T4 GPU)\n')
print(f'{"Texto":<60} {"Áudio":>8} {"Geração":>10} {"RTF":>6}')
print('-' * 90)

for txt in textos_benchmark:
    t0 = time.time()
    wav = m.generate(txt, language='pt', generation_config=cfg)[0]
    wav = np.asarray(wav, dtype=np.float32).squeeze()
    geracao = time.time() - t0
    duracao_audio = len(wav) / SR
    rtf = geracao / duracao_audio  # < 1 significa mais rápido que tempo real
    
    resumo = txt[:57] + '...' if len(txt) > 57 else txt
    print(f'{resumo:<60} {duracao_audio:>6.1f}s {geracao:>8.1f}s {rtf:>6.2f}x')

print('\nRTF < 1.0 = mais rápido que tempo real (ótimo!)')
print('RTF = 0.3 = gera 1s de áudio em 0.3s (3× mais rápido que real)')

## Célula 8 — Avaliar OmniVoice Studio vs nosso Estúdio

O OmniVoice Studio (debpalash) é um app desktop Tauri. No Colab não podemos rodá-lo,
mas podemos avaliar o que ele oferece versus o que já temos.

In [ ]:
comparacao = '''
╔══════════════════════════════════════════════════════════════════════════╗
║       COMPARAÇÃO: OmniVoice Studio vs Estúdio de Voz da Pandora        ║
╠══════════════════════════════════╦══════════════════╦═══════════════════╣
║ Feature                          ║ OmniVoice Studio ║ Estúdio Pandora   ║
╠══════════════════════════════════╬══════════════════╬═══════════════════╣
║ TTS básico                       ║ ✅               ║ ✅                ║
║ Clonagem de voz                  ║ ✅               ║ ✅                ║
║ Design de voz (instruct)         ║ ✅               ║ ✅                ║
║ LLM integrado (Gemma)            ║ ❌               ║ ✅                ║
║ 21 emoções com DSP               ║ ❌               ║ ✅                ║
║ Meditação guiada                 ║ ❌               ║ ✅                ║
║ Contador de histórias            ║ ❌               ║ ✅                ║
║ Comédia (estilo auditório)       ║ ❌               ║ ✅                ║
║ Game Show Educativo (PDF)        ║ ❌               ║ ✅                ║
║ Batalha de vozes                 ║ ❌               ║ ✅                ║
║ Rádio Pandora completo           ║ ❌               ║ ✅                ║
║ Personagem com memória           ║ ❌               ║ ✅                ║
║ Trailer cinematográfico          ║ ❌               ║ ✅                ║
║ Interface web (browser)          ║ ❌ (só desktop)   ║ ✅ (FastAPI)      ║
║ Roda no Windows                  ║ ⚠️ (instalação   ║ ✅ (pip install)  ║
║                                  ║ complexa Tauri)  ║                   ║
║ Suporte a pt-BR como 1ª classe   ║ ⚠️ (en-US first) ║ ✅                ║
║ Ciclo VRAM (Gemma↔OmniVoice)    ║ N/A              ║ ✅ (RTX 2060 6GB) ║
╠══════════════════════════════════╬══════════════════╬═══════════════════╣
║ VEREDICTO                        ║ UI bonita,       ║ Muito mais        ║
║                                  ║ TTS puro         ║ funcionalidades   ║
╚══════════════════════════════════╩══════════════════╩═══════════════════╝

CONCLUSÃO: NÃO vale a pena instalar o OmniVoice Studio localmente.
O Estúdio da Pandora supera em todas as dimensões que importam para o projeto.
'''
print(comparacao)

## Célula 9 — Baixar todos os áudios gerados

In [ ]:
import os, zipfile
from google.colab import files

wavs = [f for f in os.listdir('.') if f.endswith('.wav')]
print(f'Arquivos WAV gerados: {wavs}')

with zipfile.ZipFile('omnivoice_testes.zip', 'w') as z:
    for wav in wavs:
        z.write(wav)

print('\n📦 Fazendo download do ZIP com todos os áudios...')
files.download('omnivoice_testes.zip')

---
## Resumo do que foi testado

| Célula | Teste | O que avaliar |
|--------|-------|---------------|
| 4 | TTS básico em pt-BR | Qualidade geral da voz, pronúncia |
| 5 | Instruct com 5 perfis | Variedade de vozes, obediência ao instruct |
| 6 | Clonagem de voz | Fidelidade ao timbre de referência |
| 7 | Benchmark de velocidade | RTF na T4 vs RTX 2060 local |
| 8 | Comparação de features | OmniVoice Studio vs Estúdio Pandora |

**Próximo passo:** ouvir os áudios baixados e decidir se algum ajuste de parâmetros
(num_step, guidance_scale, speed) vale a pena trazer para o motor local.